In [0]:
---------------------------------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`silver`.`pedidos_autoloader` AS
    SELECT
        CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        CAST(quantidade AS INT) AS quantidade,
        CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_TIMESTAMP(data_pedido) AS data_pedido
    FROM `capgemini_academy`.`bronze`.`pedidos_autoloader`
    WHERE 1=1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;
---------------------------------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`gold`.`receita_diaria` AS
    SELECT
        DATE(data_pedido) AS data_pedido,
        COUNT(DISTINCT pedido_id) AS pedidos,
        SUM(quantidade) AS itens_vendidos,
        SUM(valor_total) AS receita_total,
        ROUND(AVG(valor_total), 2) AS ticket_medio
    FROM `capgemini_academy`.`silver`.`pedidos_autoloader`
    GROUP BY DATE(data_pedido);
---------------------------------------------------------
SELECT * FROM `capgemini_academy`.`silver`.`pedidos_autoloader`;

SELECT * FROM `capgemini_academy`.`gold`.`receita_diaria`;
---------------------------------------------------------
MERGE INTO `capgemini_academy`.`silver`.`pedidos_autoloader` AS target
USING `capgemini_academy`.`bronze`.`pedidos_autoloader` AS source
    ON target.pedido_id = source.pedido_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
---------------------------------------------------------
